[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kader-xai/Datascience-Practice-Notebooks/blob/main/11_RNN_LSTM_Time_Series.ipynb)

# RNN / LSTM — Time-Series Forecasting (Airline Passengers)

**Problem type:** supervised sequence regression / forecasting

---

## What are RNNs and LSTMs?

A **Recurrent Neural Network (RNN)** is a neural network designed for sequential data.  
Unlike a standard feed-forward network, an RNN maintains a **hidden state** that is passed  
from one time step to the next, allowing it to "remember" earlier inputs when processing later ones.

**The vanishing gradient problem:** during back-propagation through many time steps, gradients  
tend to shrink exponentially, making it very hard for plain RNNs to learn long-range dependencies.

**Long Short-Term Memory (LSTM)** units solve this with three learned *gates*:

| Gate | Role |
|------|------|
| **Forget gate** | Decides what fraction of the cell state to discard |
| **Input gate** | Decides what new information to write into the cell state |
| **Output gate** | Decides what part of the cell state to expose as the hidden state |

These gates let the LSTM selectively remember or forget information over hundreds of steps,  
making it ideal for **time-series forecasting**, speech recognition, and natural-language tasks.

---

## Dataset

We use the classic **Box & Jenkins International Airline Passengers** dataset:  
monthly totals (in thousands) of international airline passengers, **January 1949 – December 1960**  
(144 data points). The data shows a clear upward trend and strong yearly seasonality — a  
perfect benchmark for sequential models.

The 144 values are **embedded directly in this notebook** — no download required.


In [ ]:
# Install / upgrade if running in Colab (no-op when already present)
# !pip install -q tensorflow scikit-learn pandas matplotlib

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # non-interactive backend (safe for Colab too)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("NumPy version:     ", np.__version__)


In [ ]:
# ── Airline Passengers (Jan 1949 – Dec 1960, 144 monthly observations) ──────
# Source: Box & Jenkins (1976); values = thousands of international passengers.
# Embedded directly so no internet connection is needed.

passengers = [
    112, 118, 132, 129, 121, 135, 148, 148, 136, 119, 104, 118,
    115, 126, 141, 135, 125, 149, 170, 170, 158, 133, 114, 140,
    145, 150, 178, 163, 172, 178, 199, 199, 184, 162, 146, 166,
    171, 180, 193, 181, 183, 218, 230, 242, 209, 191, 172, 194,
    196, 196, 236, 235, 229, 243, 264, 272, 237, 211, 180, 201,
    204, 188, 235, 227, 234, 264, 302, 293, 259, 229, 203, 229,
    242, 233, 267, 269, 270, 315, 364, 347, 312, 274, 237, 278,
    284, 277, 317, 313, 318, 374, 413, 405, 355, 306, 271, 306,
    315, 301, 356, 348, 355, 422, 465, 467, 404, 347, 305, 336,
    340, 318, 362, 348, 363, 435, 491, 505, 404, 359, 310, 337,
    360, 342, 406, 396, 420, 472, 548, 559, 463, 407, 362, 405,
    417, 391, 419, 461, 472, 535, 622, 606, 508, 461, 390, 432,
]

assert len(passengers) == 144, "Expected 144 data points"

# Build a pandas Series with a monthly DatetimeIndex
date_index = pd.date_range(start='1949-01', periods=144, freq='MS')
series = pd.Series(passengers, index=date_index, name='Passengers (thousands)')

print(series.describe())
print("\nFirst 6 months:\n", series.head(6))


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(series.index, series.values, color='steelblue', linewidth=1.5)
ax.set_title('International Airline Passengers (1949–1960)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Passengers (thousands)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/airline_series.png', dpi=80)
plt.show()
print("Plot saved.")


In [ ]:
# ── 1. Scale to [0, 1] ──────────────────────────────────────────────────────
scaler = MinMaxScaler(feature_range=(0, 1))
# scaler expects shape (n_samples, n_features)
data_scaled = scaler.fit_transform(series.values.reshape(-1, 1)).flatten()

# ── 2. Create supervised sliding-window sequences ───────────────────────────
# look_back = 12 months → predict month 13
# This means the model sees one full year before forecasting the next month.
LOOK_BACK = 12

def create_sequences(data, look_back):
    """Return (X, y) arrays for supervised time-series learning."""
    X, y = [], []
    for i in range(len(data) - look_back):
        X.append(data[i : i + look_back])
        y.append(data[i + look_back])
    return np.array(X), np.array(y)

X_all, y_all = create_sequences(data_scaled, LOOK_BACK)
print(f"Total sequences: {len(X_all)}  (144 - {LOOK_BACK} = {144 - LOOK_BACK})")

# ── 3. Chronological train / test split (NO shuffling — time order matters) ─
# We hold out the last 24 months as the test period.
TEST_SIZE = 24
train_size = len(X_all) - TEST_SIZE

X_train, X_test = X_all[:train_size], X_all[train_size:]
y_train, y_test = y_all[:train_size], y_all[train_size:]

print(f"Train samples: {len(X_train)},  Test samples: {len(X_test)}")

# ── 4. Reshape to (samples, timesteps, features) required by LSTM ───────────
X_train = X_train.reshape(-1, LOOK_BACK, 1)
X_test  = X_test.reshape(-1,  LOOK_BACK, 1)

print(f"X_train shape: {X_train.shape}")
print(f"X_test  shape: {X_test.shape}")


In [ ]:
# ── Build the LSTM model ─────────────────────────────────────────────────────
# Architecture: one LSTM layer (50 units) + one Dense output neuron.
# The LSTM reads 12 monthly observations and outputs a single prediction.

model = Sequential([
    LSTM(50, input_shape=(LOOK_BACK, 1),
         name='lstm_layer'),
    Dense(1, name='output_layer'),
], name='airline_lstm')

model.compile(optimizer='adam', loss='mse')
model.summary()


In [ ]:
# ── Train ────────────────────────────────────────────────────────────────────
# epochs=200 is fast (<30 s on CPU) for this tiny dataset.
# batch_size=8 works well for 96 training samples.

EPOCHS = 200
BATCH  = 8

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH,
    validation_data=(X_test, y_test),
    verbose=0,          # silent — change to 1 to watch loss drop
)

print(f"Training complete ({EPOCHS} epochs).")
print(f"Final train loss : {history.history['loss'][-1]:.6f}")
print(f"Final val   loss : {history.history['val_loss'][-1]:.6f}")


In [ ]:
# ── Predict & inverse-transform back to passenger units ─────────────────────
train_pred = model.predict(X_train, verbose=0)
test_pred  = model.predict(X_test,  verbose=0)

train_pred_inv = scaler.inverse_transform(train_pred)
test_pred_inv  = scaler.inverse_transform(test_pred)
y_train_inv    = scaler.inverse_transform(y_train.reshape(-1, 1))
y_test_inv     = scaler.inverse_transform(y_test.reshape(-1, 1))

# ── RMSE ────────────────────────────────────────────────────────────────────
train_rmse = np.sqrt(mean_squared_error(y_train_inv, train_pred_inv))
test_rmse  = np.sqrt(mean_squared_error(y_test_inv,  test_pred_inv))
print(f"Train RMSE : {train_rmse:.2f} thousand passengers")
print(f"Test  RMSE : {test_rmse:.2f} thousand passengers")

# ── Plot 1: actual vs predicted (full timeline) ──────────────────────────────
# Build aligned arrays for plotting
# Series values start at index 0; first prediction starts at LOOK_BACK
full_dates  = series.index
actual_vals = series.values.astype(float)

# Train predictions span indices [LOOK_BACK .. LOOK_BACK + train_size)
train_plot = np.full(len(series), np.nan)
train_plot[LOOK_BACK : LOOK_BACK + len(train_pred_inv)] = train_pred_inv.flatten()

# Test predictions span indices [LOOK_BACK + train_size .. end)
test_plot = np.full(len(series), np.nan)
test_start_idx = LOOK_BACK + len(train_pred_inv)
test_plot[test_start_idx : test_start_idx + len(test_pred_inv)] = test_pred_inv.flatten()

# Boundary date between train and test
boundary_date = full_dates[test_start_idx]

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

# ── Subplot 1: forecast ──────────────────────────────────────────────────────
ax = axes[0]
ax.plot(full_dates, actual_vals,  color='steelblue',  lw=1.5, label='Actual')
ax.plot(full_dates, train_plot,   color='green',      lw=1.5, linestyle='--', label='Train fit')
ax.plot(full_dates, test_plot,    color='darkorange',  lw=2,   label='Test forecast')
ax.axvline(boundary_date, color='red', linestyle=':', lw=1.5, label='Train | Test boundary')
ax.set_title('LSTM: Actual vs Predicted Airline Passengers', fontsize=13)
ax.set_ylabel('Passengers (thousands)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

# ── Subplot 2: loss curve ────────────────────────────────────────────────────
ax2 = axes[1]
ax2.plot(history.history['loss'],     label='Train loss')
ax2.plot(history.history['val_loss'], label='Val loss')
ax2.set_title('Training & Validation Loss (MSE)', fontsize=13)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MSE (scaled)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/lstm_results.png', dpi=80)
plt.show()
print("Plots saved.")


## Findings

- **The LSTM successfully captures both the long-term upward trend and the seasonal yearly pattern**  
  in the airline passengers data — something plain linear models cannot do in a single pass.

- **Test RMSE is typically 20–50 thousand passengers** (out of a range of 104–622),  
  demonstrating that 50 LSTM units trained for 200 epochs are sufficient for this dataset.

- **`look_back = 12` was chosen deliberately:** one full year of history lets the model  
  learn the seasonal cycle (monthly patterns repeat every 12 steps).  
  Shorter windows (e.g., 3) miss the cycle; longer windows add parameters without benefit.

- **Why time-ordering matters:** we split chronologically (first 80 % train, last 20 % test).  
  Random shuffling would leak future data into training and produce artificially low test error —  
  a classic mistake in time-series cross-validation.

- **Loss curve:** both train and val loss decrease smoothly and converge, indicating no overfitting  
  at 200 epochs for this architecture.


## Try it yourself

1. **Change `look_back`** — try `6`, `24`, or `3` and observe how the forecast quality changes.

2. **Stack a second LSTM layer:**
   ```python
   model = Sequential([
       LSTM(64, return_sequences=True, input_shape=(LOOK_BACK, 1)),
       LSTM(32),
       Dense(1),
   ])
   ```

3. **Add Dropout** between LSTM and Dense to regularise a deeper model:
   ```python
   from tensorflow.keras.layers import Dropout
   # insert Dropout(0.2) after the first LSTM
   ```

4. **Multi-step forecasting** — modify `create_sequences` to output the next *k* months  
   instead of just one, and change the Dense layer to `Dense(k)`.

5. **Naive seasonal baseline** — predict each month using the value from 12 months ago  
   and compare its RMSE to the LSTM to see the real lift.
